# MiniMind Pretrain vs Full SFT 推理对比

这个 notebook 用同一个 prompt 分别测试 `out/pretrain_768.pth` 和 `out/full_sft_768.pth`。  
它复用了 `eval_llm.py` 的核心逻辑：pretrain 用 `bos_token + prompt`，SFT 用 tokenizer 的 chat template。

In [2]:
from pathlib import Path
import gc
import sys
import time
import random

import torch
from transformers import AutoTokenizer

ROOT = Path.cwd()
if not (ROOT / "eval_llm.py").exists():
    ROOT = Path("/home/anhuang/minimind")

sys.path.insert(0, str(ROOT))

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from trainer.trainer_utils import get_model_params, setup_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HIDDEN_SIZE = 768
NUM_HIDDEN_LAYERS = 8
USE_MOE = False

print("ROOT:", ROOT)
print("DEVICE:", DEVICE)
print("pretrain exists:", (ROOT / "out" / "pretrain_768.pth").exists())
print("full_sft exists:", (ROOT / "out" / "full_sft_768.pth").exists())

ROOT: /home/anhuang/minimind
DEVICE: cuda
pretrain exists: True
full_sft exists: True


## 工具函数

In [3]:
def clear_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_minimind(weight: str, device: str = DEVICE):
    tokenizer = AutoTokenizer.from_pretrained(ROOT / "model")
    config = MiniMindConfig(
        hidden_size=HIDDEN_SIZE,
        num_hidden_layers=NUM_HIDDEN_LAYERS,
        use_moe=USE_MOE,
    )
    model = MiniMindForCausalLM(config)
    ckpt = ROOT / "out" / f"{weight}_{HIDDEN_SIZE}.pth"
    state = torch.load(ckpt, map_location="cpu")
    model.load_state_dict(state, strict=True)
    del state
    get_model_params(model, model.config)
    model = model.eval().to(device)
    if device.startswith("cuda"):
        model = model.half()
    return model, tokenizer


@torch.inference_mode()
def generate_reply(
    model,
    tokenizer,
    prompt: str,
    weight: str,
    max_new_tokens: int = 256,
    temperature: float = 0.85,
    top_p: float = 0.95,
    seed: int | None = None,
):
    if seed is None:
        seed = random.randint(0, 31415926)
    setup_seed(seed)

    if "pretrain" in weight:
        input_text = tokenizer.bos_token + prompt
    else:
        messages = [{"role": "user", "content": prompt}]
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            open_thinking=False,
        )

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True).to(model.device)
    start = time.time()
    output_ids = model.generate(
        inputs=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        max_new_tokens=max_new_tokens,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        top_p=top_p,
        temperature=temperature,
        repetition_penalty=1.0,
    )
    new_tokens = output_ids[0][len(inputs["input_ids"][0]):]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    elapsed = time.time() - start
    speed = len(new_tokens) / max(elapsed, 1e-6)
    return response, speed, seed

## 输入一个 prompt

In [16]:
# prompt = input("请输入 prompt: ").strip()
# if not prompt:
#     prompt = "为什么天空是蓝色的？"

prompt = "moon的意思是什么"
print("Prompt:", prompt)

Prompt: moon的意思是什么


## Block 1: 只预训练后的模型

In [17]:
clear_model()
model, tokenizer = load_minimind("pretrain")
pretrain_response, pretrain_speed, seed = generate_reply(
    model,
    tokenizer,
    prompt,
    weight="pretrain",
    max_new_tokens=256,
)
print("[pretrain]")
print(pretrain_response)
print(f"\nseed={seed}, speed={pretrain_speed:.2f} tokens/s")
clear_model(model)

Model Params: 63.91M
[pretrain]
？"moon" 的意思是“玛玛”。它通常用来指某物，即人类或动物的形象。它指的是一种美丽、纯洁、神秘的特质，常常被用来形容人或物的某些品质或特质。

seed=21455603, speed=126.51 tokens/s


## Block 2: SFT 后的模型

In [18]:
clear_model()
model, tokenizer = load_minimind("full_sft")
sft_response, sft_speed, seed = generate_reply(
    model,
    tokenizer,
    prompt,
    weight="full_sft",
    max_new_tokens=256,
)
print("[full_sft]")
print(sft_response)
print(f"\nseed={seed}, speed={sft_speed:.2f} tokens/s")
clear_model(model)

Model Params: 63.91M
[full_sft]
"Moon" 是一种广泛使用的在线编程语言，通常用于创建软件、脚本或其他软件。在软件开发中，Moon 是一种用于构建、测试、设计和构建应用的工具。以下是 Moon 的意思：

1. **Moon**：
   - 在软件开发中，Moon 提供了一种结构化、可视化的编程环境，使软件能够理解和解释复杂的数据结构和功能。
   - Moon 提供了一个对象或插件，使开发者能够以图表、脚本或特定的交互方式构建和部署应用程序。

2. **Moon**：
   - 这个术语通常用于表示程序或系统结构的名称。在软件开发中，Moon 是一个常用的类。在软件开发中，它通常用于定义、定义和操作组件，以确保代码的可读性和可维护性。
   - 它可能包括代码片段、类、模块等的结构化元素。

3. **Moon**：
   - 这个术语在软件开发中是一个基本的概念，特别是在创建和测试软件或系统时。在软件开发中，它常用于创建、测试和部署应用程序

seed=14621791, speed=124.09 tokens/s


## 可选：用固定 seed 对比同一个 prompt

如果你希望减少随机采样带来的差异，可以给两个模型使用同一个 seed。

In [19]:
fixed_seed = 42

clear_model()
model, tokenizer = load_minimind("pretrain")
a, _, _ = generate_reply(model, tokenizer, prompt, "pretrain", seed=fixed_seed)
clear_model(model)

model, tokenizer = load_minimind("full_sft")
b, _, _ = generate_reply(model, tokenizer, prompt, "full_sft", seed=fixed_seed)
clear_model(model)

print("========== pretrain ==========")
print(a)
print("\n========== full_sft ==========")
print(b)

Model Params: 63.91M
Model Params: 63.91M
========== pretrain ==========
？"Moon" 通常在不同的语境中表示不同的意思。在中文中，这个词是"Moon"，在不同的语境中可能代表不同的含义。

========== full_sft ==========
"Moon"在中文中，意思是 "moon" 或 "moon"。"Moon" 在中文中是表示“一种器官”或“一种器官”的术语，通常用于指代一种器官或其他器官。

"moon" 可以用来描述指代一种器官的外在特征或功能。例如，它可能是描述一个人的体型、身体的结构或功能的外观。在不同的上下文中，"Moon" 可以有不同的含义。

至于 "Moon" 这个词，其实有一些类似的含义。例如，“moon” 可能指的是“指令”，在不同的语境下，“Moon” 可以指代某种方法或工具。

"Moon" 通常指的是一个特定的术语或概念，用于描述某种器官或系统内部的结构或功能。它通常用于描述特定的系统或结构，如人体、植物、动物等。

"Moon" 通常指的是“指令”或“结构”，表示某个系统或系统的结构或功能。它在不同的语境中有
